# Commentary Ingestion Pipeline v3
## Key change from v2: local YouTube cache replaces per-race API searches

**Quota usage comparison**
- v2: 4 channels × 100 units × 90 races = **36,000 units/day** (exhausted after ~25 races)
- v3: one-time cache download ≈ **few hundred units total**, then zero quota for all matching

**Daily workflow**
1. **Cell 3** — Refresh local YouTube cache (run weekly, or after new races air)
2. **Cell 5** — `run_automated_search()` — local matching, no quota cost, process all 1,000 races in one run
3. **Cell 6** — `run_transcript_fetch()` — unchanged from v2
4. **Cell 7** — Manual override (unchanged)
5. **Cell 8** — Status dashboard (unchanged)
6. **Cell 10** — Claude tactical extraction (unchanged)
7. **Cell 11** — Commentary retrieval (unchanged)


In [23]:
import os
import re
import json
import time
import random
import datetime
from pathlib import Path
from dotenv import load_dotenv

import pandas as pd
from googleapiclient.discovery import build
from googleapiclient.errors import HttpError
from youtube_transcript_api import (
    YouTubeTranscriptApi,
    NoTranscriptFound,
    TranscriptsDisabled,
    VideoUnavailable,
)
from rapidfuzz import fuzz
import anthropic

load_dotenv()

PELOTON_YOUTUBE_API_KEY = os.getenv('PELOTON_YOUTUBE_API_KEY')
PROCESSED_DATA  = Path('../data/processed')
RAW_DIR         = Path('../data/commentary/raw')
EXTRACTED_DIR   = Path('../data/commentary/extracted')
CACHE_DIR       = Path('../data/commentary/cache')

RAW_DIR.mkdir(parents=True, exist_ok=True)
EXTRACTED_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

CACHE_FILE = CACHE_DIR / 'all_channel_videos.parquet'

ytt     = YouTubeTranscriptApi()
youtube = build('youtube', 'v3', developerKey=PELOTON_YOUTUBE_API_KEY)
claude  = anthropic.Anthropic()

print(f'YouTube API key: {"SET" if PELOTON_YOUTUBE_API_KEY else "MISSING"}')
print(f'Raw dir:         {RAW_DIR}')
print(f'Extracted dir:   {EXTRACTED_DIR}')
print(f'Cache dir:       {CACHE_DIR}')
print(f'Raw files:       {len(list(RAW_DIR.glob("*.json")))}')
print(f'Extracted files: {len(list(EXTRACTED_DIR.glob("*.json")))}')
print(f'Cache exists:    {CACHE_FILE.exists()}')


YouTube API key: SET
Raw dir:         ..\data\commentary\raw
Extracted dir:   ..\data\commentary\extracted
Cache dir:       ..\data\commentary\cache
Raw files:       897
Extracted files: 345
Cache exists:    True


In [24]:
CHANNEL_REGISTRY = [
            {"id": "UCqZQlzSHbVJrwrn5XvzrzcA", "name": "NBC Sports",         "coverage": "Grand Tours, Monuments, WorldTour"},
            {"id": "UCu7phdCr-raU7OaJfEpHZww", "name": "GCN Racing",          "coverage": "All WorldTour"},
            {"id": "UCfDfvvMARk4TKcC62ALi6eA", "name": "TNT Sports Cycling",  "coverage": "Grand Tours, European classics"},
            # {"id": "UCuTaETsuCOkJ0H_GAztWt0Q", "name": "GCN",                 "coverage": "All WorldTour"},
            # Official race channels
            {"id": "UCSpycUnuU0IVF7gGIhGojhg", "name": "Tour de France",      "coverage": "Tour de France official"},
            {"id": "UCe10BxbsFg9Kbmkg-ean_Dg", "name": "Giro d'Italia",       "coverage": "Giro d'Italia official"},
            {"id": "UCf7iHZIcKEhiN34-fETtNCA", "name": "La Vuelta",           "coverage": "Vuelta a España official"},
            # {"id": "UCozt5iXNqmhU1I7tcjJ0UFQ", "name": "Eurosport",    "coverage": "All WorldTour"},
            {"id": "UCm0Qjs5OBrv3-d6kKBshEbg", "name": "Tour Down Under",     "coverage": "Tour Down Under official"},
            {"id": "UCXgba6tOLghtJuXaD8LBHWg", "name": "inCycle",             "coverage": "All WorldTour"},
            {"id": "UCcbBlBEtCZ2lX7bodgi02Xg", "name": "Velon",               "coverage": "All WorldTour"},
            {"id": "UClhp9g6TPiqCTOlcw0ROfNg", "name": "TNT Sports",          "coverage": "All WorldTour"},
            
        ]
print(f'Channels: {[c["name"] for c in CHANNEL_REGISTRY]}')


Channels: ['NBC Sports', 'GCN Racing', 'TNT Sports Cycling', 'Tour de France', "Giro d'Italia", 'La Vuelta', 'Tour Down Under', 'inCycle', 'Velon', 'TNT Sports']


In [25]:
# ── CELL 3: Build / refresh local YouTube cache ─────────────────────────────
# Run once to download all uploads from the four channels.
# After that, run weekly (or after a race airs) to pick up new videos.
# Quota cost: ~1–2 units per 50 videos fetched — effectively free.

def _get_upload_playlist(channel_id: str) -> str | None:
    """Get the uploads playlist ID for a channel. Costs 1 quota unit."""
    resp = youtube.channels().list(part='contentDetails', id=channel_id).execute()
    items = resp.get('items', [])
    if not items:
        return None
    return items[0]['contentDetails']['relatedPlaylists']['uploads']


def _fetch_playlist_page(playlist_id: str, page_token=None):
    """Fetch one page of playlist items. Costs 1 quota unit."""
    kwargs = dict(part='snippet', playlistId=playlist_id, maxResults=50)
    if page_token:
        kwargs['pageToken'] = page_token
    return youtube.playlistItems().list(**kwargs).execute()


def fetch_channel_videos(channel: dict, max_pages: int = 500) -> pd.DataFrame:
    """
    Download all video metadata from a channel's uploads playlist.
    max_pages=500 → up to 25,000 videos per channel.
    """
    playlist_id = _get_upload_playlist(channel['id'])
    if not playlist_id:
        print(f'  No uploads playlist found for {channel["name"]}')
        return pd.DataFrame()

    rows, page_token, pages = [], None, 0
    while pages < max_pages:
        resp       = _fetch_playlist_page(playlist_id, page_token)
        pages     += 1
        for item in resp.get('items', []):
            s = item['snippet']
            vid_id = s.get('resourceId', {}).get('videoId')
            if not vid_id or s.get('title') == 'Private video':
                continue
            rows.append({
                'video_id':  vid_id,
                'title':     s['title'],
                'published': s['publishedAt'],
                'channel':   channel['name'],
                'channel_id': channel['id'],
            })
        page_token = resp.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.1)

    df = pd.DataFrame(rows)
    if not df.empty:
        df['published'] = pd.to_datetime(df['published'], utc=True)
    return df


def build_video_cache(force_refresh: bool = False) -> pd.DataFrame:
    """
    Download all channel uploads and save to parquet.
    Safe to re-run — loads from disk unless force_refresh=True.
    """
    if CACHE_FILE.exists() and not force_refresh:
        df = pd.read_parquet(CACHE_FILE)
        age_hrs = (datetime.datetime.utcnow() -
                   datetime.datetime.fromtimestamp(CACHE_FILE.stat().st_mtime)).seconds / 3600
        print(f'Cache loaded: {len(df):,} videos (last updated {age_hrs:.1f}h ago)')
        print(f'Run build_video_cache(force_refresh=True) to re-download')
        return df

    print('Downloading channel uploads — this takes a few minutes...')
    all_dfs = []
    for ch in CHANNEL_REGISTRY:
        print(f'  {ch["name"]}...')
        ch_df = fetch_channel_videos(ch)
        print(f'    → {len(ch_df):,} videos')
        all_dfs.append(ch_df)
        time.sleep(1)

    df = pd.concat(all_dfs, ignore_index=True)
    df.to_parquet(CACHE_FILE)
    print(f'\nCache saved: {len(df):,} videos → {CACHE_FILE}')
    return df


def refresh_recent(days_back: int = 30) -> pd.DataFrame:
    """
    Lightweight refresh: re-fetch only the first few pages of each channel
    to pick up new uploads without re-downloading the entire history.
    Costs ~8 quota units total (one per channel × 2 pages).
    """
    if not CACHE_FILE.exists():
        print('No cache found — run build_video_cache() first')
        return pd.DataFrame()

    existing = pd.read_parquet(CACHE_FILE)
    cutoff   = pd.Timestamp.utcnow() - pd.Timedelta(days=days_back)
    new_rows = []

    for ch in CHANNEL_REGISTRY:
        playlist_id = _get_upload_playlist(ch['id'])
        if not playlist_id:
            continue
        page_token = None
        for _ in range(4):   # ≤4 pages = 200 newest videos per channel
            resp = _fetch_playlist_page(playlist_id, page_token)
            for item in resp.get('items', []):
                s = item['snippet']
                vid_id = s.get('resourceId', {}).get('videoId')
                if not vid_id:
                    continue
                pub = pd.to_datetime(s['publishedAt'], utc=True)
                if pub < cutoff:
                    break
                if vid_id not in existing['video_id'].values:
                    new_rows.append({'video_id': vid_id, 'title': s['title'],
                                     'published': pub, 'channel': ch['name'],
                                     'channel_id': ch['id']})
            page_token = resp.get('nextPageToken')
            if not page_token:
                break
        time.sleep(0.5)

    if new_rows:
        fresh = pd.DataFrame(new_rows)
        merged = pd.concat([existing, fresh], ignore_index=True).drop_duplicates('video_id')
        merged.to_parquet(CACHE_FILE)
        print(f'Refresh complete: added {len(new_rows)} new videos ({len(merged):,} total)')
        return merged
    else:
        print(f'Refresh complete: no new videos found')
        return existing


# ── RUN ONCE (or weekly) ──────────────────────────────────────────────────────
# First time: downloads everything (~few hundred quota units)
# After that: loads from disk instantly
all_videos = build_video_cache()


Cache loaded: 65,984 videos (last updated 5.5h ago)
Run build_video_cache(force_refresh=True) to re-download


C:\Users\Admin\AppData\Local\Temp\ipykernel_18196\992991223.py:67: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  age_hrs = (datetime.datetime.utcnow() -


In [22]:
# ── CELL: Fetch older NBC Sports videos ──────────────────────────────────────
# NBC Sports has ~44,000 total videos but the standard playlist pull
# only captures the most recent ~20,000 (400 pages × 50 results).
# This cell fetches the older half by skipping to page 400 and collecting
# from there onwards.
#
# Quota cost: ~900 units total (400 skip + 500 collect × 1 unit/page)
# Runtime:    ~5-8 minutes (mostly the skip phase)
#
# Run ONCE after your initial build_video_cache() call.
# Safe to re-run — drop_duplicates handles any overlap.

def fetch_nbc_older(skip_pages: int = 400) -> pd.DataFrame:
    """
    Fetch older NBC Sports videos by skipping past the most recent pages.

    The standard playlist pull gets the newest 20,000 videos (pages 1-400).
    This gets everything from page 400 onwards — the older catalog.

    Args:
        skip_pages: Pages to skip before collecting. Default 400 = skip
                    the 20,000 videos already in your cache.
    """
    nbc_channel = next(c for c in CHANNEL_REGISTRY if c['name'] == 'NBC Sports')

    # Get upload playlist
    resp = youtube.channels().list(
        part='contentDetails', id=nbc_channel['id']
    ).execute()
    playlist_id = resp['items'][0]['contentDetails']['relatedPlaylists']['uploads']

    # ── Phase 1: Skip forward ─────────────────────────────────────────────────
    print(f'Skipping {skip_pages} pages (~{skip_pages * 50:,} videos)...')
    print('This takes about 2-3 minutes. Grab a coffee.')
    page_token = None

    for i in range(skip_pages):
        resp = youtube.playlistItems().list(
            part='snippet', playlistId=playlist_id,
            maxResults=50, pageToken=page_token
        ).execute() if page_token else youtube.playlistItems().list(
            part='snippet', playlistId=playlist_id, maxResults=50
        ).execute()

        page_token = resp.get('nextPageToken')
        if not page_token:
            print(f'Playlist ended at page {i} — only {i * 50:,} videos total')
            return pd.DataFrame()
        if i % 100 == 0 and i > 0:
            print(f'  Skipped {i} pages ({i * 50:,} videos)...')
        time.sleep(0.05)

    print(f'Skip complete. Now collecting from page {skip_pages} onwards...')

    # ── Phase 2: Collect from here ────────────────────────────────────────────
    rows  = []
    pages = 0

    while pages < 500:
        kwargs = dict(part='snippet', playlistId=playlist_id, maxResults=50)
        if page_token:
            kwargs['pageToken'] = page_token
        resp   = youtube.playlistItems().list(**kwargs).execute()
        pages += 1

        for item in resp.get('items', []):
            s = item['snippet']
            vid_id = s.get('resourceId', {}).get('videoId')
            if not vid_id or s.get('title') == 'Private video':
                continue
            rows.append({
                'video_id':   vid_id,
                'title':      s['title'],
                'published':  s['publishedAt'],
                'channel':    'NBC Sports',
                'channel_id': nbc_channel['id'],
            })

        page_token = resp.get('nextPageToken')
        if not page_token:
            break
        time.sleep(0.1)

    df = pd.DataFrame(rows)
    if not df.empty:
        df['published'] = pd.to_datetime(df['published'], utc=True)

    print(f'Collected {len(df):,} older NBC Sports videos ({pages} pages)')
    return df


def merge_nbc_older_into_cache(nbc_older_df: pd.DataFrame) -> pd.DataFrame:
    """Merge older NBC videos into the existing cache."""
    existing = pd.read_parquet(CACHE_FILE)
    print(f'Existing cache: {len(existing):,} videos')

    merged = (
        pd.concat([existing, nbc_older_df], ignore_index=True)
        .drop_duplicates('video_id')
        .reset_index(drop=True)
    )
    merged.to_parquet(CACHE_FILE)

    added = len(merged) - len(existing)
    print(f'Cache updated: {len(existing):,} → {len(merged):,} (+{added:,} new NBC videos)')
    return merged


# ── Run ───────────────────────────────────────────────────────────────────────
nbc_older = fetch_nbc_older(skip_pages=400)

if not nbc_older.empty:
    # Preview what era we're getting
    print(f'\nDate range of older videos:')
    print(f'  Oldest: {nbc_older["published"].min().date()}')
    print(f'  Newest: {nbc_older["published"].max().date()}')
    print(f'\nSample titles:')
    for title in nbc_older.head(5)['title']:
        print(f'  {title}')

    # Merge into cache
    all_videos = merge_nbc_older_into_cache(nbc_older)
    print(f'\nCache now has {len(all_videos):,} videos total')

Skipping 400 pages (~20,000 videos)...
This takes about 2-3 minutes. Grab a coffee.
  Skipped 100 pages (5,000 videos)...
  Skipped 200 pages (10,000 videos)...
  Skipped 300 pages (15,000 videos)...
Playlist ended at page 399 — only 19,950 videos total


In [26]:
# ── CELL 4: Helper functions + race index ────────────────────────────────────
def parse_race_name_and_stage(race_name_full: str):
    clean       = re.sub(r'^\d{4}\s+', '', race_name_full).strip()
    stage_match = re.search(r'Stage\s+(\d+)', clean, re.IGNORECASE)
    stage       = int(stage_match.group(1)) if stage_match else None
    race        = re.sub(r'\s*Stage\s+\d+', '', clean, flags=re.IGNORECASE).strip()
    return race, stage


def make_label(race_name, race_date, stage):
    label = f'{str(race_date)[:4]} {race_name}'
    return label + (f' Stage {stage}' if stage else '')


def make_safe_name(label):
    return re.sub(r'[^a-z0-9]+', '_', label.lower()).strip('_')


def fetch_transcript(video_id: str) -> dict:
    try:
        transcript = ytt.fetch(video_id)
        raw_text   = ' '.join([s.text for s in transcript])
        clean_text = re.sub(r'\[.*?\]', '', raw_text)
        clean_text = re.sub(r'\(.*?\)', '', clean_text)
        clean_text = re.sub(r'\s+', ' ', clean_text).strip()
        duration   = round(transcript[-1].start, 0) if transcript else 0
        return {
            'success':       True,
            'video_id':      video_id,
            'snippet_count': len(transcript),
            'raw_chars':     len(raw_text),
            'clean_chars':   len(clean_text),
            'duration_mins': round(duration / 60, 1),
            'clean_text':    clean_text,
            'preview_start': clean_text[:500],
            'preview_end':   clean_text[-500:],
            'error':         None,
        }
    except NoTranscriptFound:
        return {'success': False, 'video_id': video_id, 'error': 'no_transcript'}
    except TranscriptsDisabled:
        return {'success': False, 'video_id': video_id, 'error': 'transcripts_disabled'}
    except VideoUnavailable:
        return {'success': False, 'video_id': video_id, 'error': 'video_unavailable'}
    except Exception as e:
        return {'success': False, 'video_id': video_id, 'error': str(e)}


def save_no_video(label, race_name, race_date, stage, reason='no_video_found') -> Path:
    safe_name = make_safe_name(label)
    out_path  = RAW_DIR / f'{safe_name}.json'
    result = {
        'label':      label,
        'race_name':  race_name,
        'race_date':  str(race_date)[:10],
        'stage':      stage,
        'video':      None,
        'transcript': None,
        'status':     reason,
    }
    with open(out_path, 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)
    return out_path


# Build race index — sort newest-first so recent (more likely to have transcripts) run first
merged_df = pd.read_csv(PROCESSED_DATA / 'merged_uci_races.csv', low_memory=False)
merged_df['Date'] = pd.to_datetime(merged_df['Date'])

race_index = (
    merged_df[['Race Name', 'Race_results', 'Date', 'Year_results', 'Stage_results']]
    .drop_duplicates('Race Name')
    .sort_values('Date', ascending=False)   # ← newest first (more likely to have transcripts/videos)
    .reset_index(drop=True)
)

raw_files   = list(RAW_DIR.glob('*.json'))
saved_names = {f.stem for f in raw_files}

statuses = {}
for path in raw_files:
    with open(path, encoding='utf-8') as f:
        data = json.load(f)
    s = data.get('status', 'unknown')
    statuses[s] = statuses.get(s, 0) + 1

transcript_saved = statuses.get('transcript_saved', 0)
no_video         = statuses.get('no_video_found', 0)
video_found      = statuses.get('video_found', 0)
ip_errors        = sum(v for k, v in statuses.items() if 'block' in k.lower() or 'ip' in k.lower())
other_errors     = sum(v for k, v in statuses.items()
                       if k not in ['transcript_saved', 'no_video_found', 'video_found', 'unknown']
                       and 'block' not in k.lower() and 'ip' not in k.lower())
remaining        = len(race_index) - len(raw_files)

print(f'Total races in dataset:   {len(race_index):,}')
print(f'Already processed:        {len(raw_files):,}')
print(f'  Transcripts saved:      {transcript_saved}')
print(f'  Video found (pending):  {video_found}')
print(f'  No video found:         {no_video}')
print(f'  IP/transcript errors:   {ip_errors + other_errors}')
print(f'Remaining (unprocessed):  {remaining}')
print(f'Est. time at 500/run:     {remaining // 500 + 1} runs (no quota limit!)')


Total races in dataset:   896
Already processed:        897
  Transcripts saved:      347
  Video found (pending):  456
  No video found:         76
  IP/transcript errors:   365
Remaining (unprocessed):  -1
Est. time at 500/run:     0 runs (no quota limit!)


In [27]:
# ── CELL 5: Local matching — zero quota cost ─────────────────────────────────
# Synced with src/peloton_iq/commentary/transcript.py

SKIP_KEYWORDS = [
    'preview', 'interview', 'training', 'behind the scenes',
    'beginner', 'guide', 'how to', 'shorts', 'power data',
    'riders to watch', 'ones to watch', 'top 10',
    'what to watch', 'race preview', 'stage preview',
    'team time trial preparation', 'kitchen', 'mechanic',
    'neutral service', 'french phrases', 'injuries',
    'dutch corner', 'how to watch',
]
WOMEN_KEYWORDS = ['femmes', 'feminine', 'women', 'ladies', 'rosa', 'giro rosa']

# Channel priority tiers — higher = better
# Official race channels (tier 4) beat generalist channels on their own race only
CHANNEL_TIER = {
    'NBC Sports':         4,
    'GCN Racing':         1,
    'TNT Sports Cycling': 2,
    # 'Eurosport':          2,
    # 'GCN':                1,
    'Tour de France':     4,   # official — TdF stages only
    "Giro d'Italia":     4,   # official — Giro stages only
    'La Vuelta':          4,   # official — Vuelta stages only
    'Tour Down Under': 4,
    'inCycle': 2,
    'Velon': 2,
    'TNT Sports': 2,
}

# Official channel → which races they cover
OFFICIAL_CHANNEL_RACES = {
    'Tour de France': ['tour de france'],
    "Giro d'Italia": ["giro d'italia", 'giro d italia', 'giro ditalia'],
    'La Vuelta':      ['vuelta a espana', 'vuelta', 'la vuelta'],
    'Tour Down Under': ['tour down under', 'santos tour down under']
}

# Known wrong-race patterns — penalize heavily when title contains these
# for races that shouldn't match them
WRONG_RACE_PATTERNS = {
    # Race name keyword → list of title patterns that indicate a DIFFERENT race
    "giro d'italia":    ['giro rosa', 'giro women', 'giro donne'],
    "tour de france":    ['tour de la provence', 'tour down under', 'tour of britain',
                         'tour de romandie', 'tour de pologne', 'tour de suisse'],
    "tour de suisse":    ['tour de france', 'tour de la provence'],
    "tour de pologne":   ['tour de france', 'tour de la provence'],
    "vuelta a espana":   ['vuelta a san juan', 'vuelta al pais vasco'],
    "tirreno-adriatico": ['giro rosa'],
    "paris-nice":        ['giro rosa'],
    "volta a catalunya": ['giro rosa'],
    "itzulia basque country": ['giro rosa'],
    "tour de romandie":  ['tour de france'],
}


def _get_channel_tier(channel: str, race_name: str) -> int:
    """Official channels only get their tier 4 bonus for their own race."""
    base_tier = CHANNEL_TIER.get(channel, 0)
    if base_tier == 4:
        race_lower = race_name.lower()
        allowed = OFFICIAL_CHANNEL_RACES.get(channel, [])
        if not any(r in race_lower for r in allowed):
            return 1
    return base_tier


def _wrong_race_penalty(race_name: str, title: str) -> float:
    """
    Penalize videos that are clearly the wrong race despite fuzzy name match.
    E.g. "Giro Rosa" matching "Giro d'Italia", "Tour de Provence" matching "Tour de France".
    """
    race_lower  = race_name.lower()
    title_lower = title.lower()
    for race_key, bad_patterns in WRONG_RACE_PATTERNS.items():
        if race_key in race_lower:
            for pattern in bad_patterns:
                if pattern in title_lower:
                    return -80.0   # strong enough to drop below any threshold
    return 0.0


def _is_nbc_priority(race_name: str) -> bool:
    rn = race_name.lower()
    return any(r in rn for r in ['tour de france', 'vuelta a espana', 'vuelta españa'])


def _is_gcn_priority(race_name: str) -> bool:
    rn = race_name.lower()
    return any(r in rn for r in [
        "giro d'italia", 'giro d italia', 'paris-roubaix', 'paris roubaix',
        'tour of flanders', 'ronde van vlaanderen', 'liege-bastogne-liege',
        'milan-san remo', 'milan san remo', 'il lombardia',
    ])


def score_local_video(row, race_name: str, race_date: str, stage=None) -> float:
    score     = 0.0
    title     = str(row['title']).lower()
    channel   = str(row.get('channel', ''))
    race_dt   = pd.to_datetime(race_date, utc=True)
    pub_dt    = pd.to_datetime(row['published'], utc=True)
    days_diff = abs((pub_dt.date() - race_dt.date()).days)

    # Wrong race hard penalty — apply first, can exit early
    wrong_penalty = _wrong_race_penalty(race_name, title)
    if wrong_penalty < 0:
        return wrong_penalty

    # Women's race penalty
    if any(k in title for k in WOMEN_KEYWORDS):
        return -80.0

    # Publish date proximity
    if days_diff == 0:      score += 60
    elif days_diff <= 1:    score += 50
    elif days_diff <= 3:    score += 35
    elif days_diff <= 7:    score += 15
    elif days_diff > 365:   score -= 40 * abs(pub_dt.year - race_dt.year)

    # Race name fuzzy match
    name_score = fuzz.partial_ratio(race_name.lower(), title)
    score += name_score * 0.25
    if name_score < 40:     score -= 40

    # Stage match
    if stage:
        score += 30 if any(p in title for p in [
            f'stage {stage}', f'stage{stage}',
            f'étape {stage}', f'tappa {stage}', f'etapa {stage}'
        ]) else -10

    # Year in title
    if str(race_date)[:4] in title:
        score += 15

    # Content type bonuses
    if 'extended highlights' in title:  score += 20
    elif 'extended' in title:           score += 12
    elif 'highlights' in title:         score += 5
    if 'full race' in title:            score += 8

    # Channel tier bonus (official channels get tier 4 only for their own race)
    score += _get_channel_tier(channel, race_name) * 5

    # NBC priority grand tours
    if _is_nbc_priority(race_name) and channel == 'NBC Sports':
        score += 20
        if 'extended highlights' in title:
            score += 25

    # Official race channel bonus
    for official_channel, races in OFFICIAL_CHANNEL_RACES.items():
        if channel == official_channel:
            race_lower = race_name.lower()
            if any(r in race_lower for r in races):
                score += 25
                if 'highlights' in title:
                    score += 15

    # GCN priority races
    if _is_gcn_priority(race_name) and channel in ('GCN Racing', 'TNT Sports Cycling', 'Eurosport'):
        score += 15

    # Skip penalties
    if any(k in title for k in SKIP_KEYWORDS):  score -= 30

    return round(score, 2)


def find_best_video_local(race_name, race_date, stage=None, threshold=75.0):
    if all_videos is None or all_videos.empty:
        return {'found': False, 'error': 'cache_empty'}

    race_year  = str(race_date)[:4]
    candidates = all_videos[
        all_videos['title'].str.contains(race_year, case=False, na=False)
    ].copy()

    if candidates.empty:
        return {'found': False}

    keywords = [w for w in race_name.split() if len(w) > 3][:2]
    if keywords:
        pattern = '|'.join(re.escape(k) for k in keywords)
        narrow  = candidates[candidates['title'].str.contains(pattern, case=False, na=False)]
        if not narrow.empty:
            candidates = narrow

    # NBC fast-path for grand tours
    if _is_nbc_priority(race_name):
        nbc = candidates[candidates['channel'] == 'NBC Sports'].copy()
        if not nbc.empty:
            nbc['match_score'] = nbc.apply(
                lambda r: score_local_video(r, race_name, race_date, stage), axis=1
            )
            nbc  = nbc.sort_values('match_score', ascending=False)
            best = nbc.iloc[0]
            if best['match_score'] >= threshold:
                return {
                    'found': True, 'video_id': best['video_id'],
                    'title': best['title'], 'published': str(best['published']),
                    'channel': best['channel'],
                    'url': f"https://youtube.com/watch?v={best['video_id']}",
                    'match_score': float(best['match_score']),
                    'all_candidates': nbc.head(5).to_dict('records'),
                }

    # Full pool scoring
    candidates['match_score'] = candidates.apply(
        lambda r: score_local_video(r, race_name, race_date, stage), axis=1
    )
    candidates = candidates.sort_values('match_score', ascending=False)
    # After scoring the full candidate pool, before returning:
    # Prefer "extended highlights" videos if one scores above threshold
    extended = candidates[
        candidates['title'].str.contains('extended highlights', case=False, na=False)
    ]
    if not extended.empty and extended.iloc[0]['match_score'] >= threshold:
        best = extended.iloc[0]
    else:
        best = candidates.iloc[0]

    if best['match_score'] < threshold:
        return {'found': False}

    return {
        'found': True, 'video_id': best['video_id'],
        'title': best['title'], 'published': str(best['published']),
        'channel': best['channel'],
        'url': f"https://youtube.com/watch?v={best['video_id']}",
        'match_score': float(best['match_score']),
        'all_candidates': candidates.head(10).to_dict('records'),
    }

In [7]:
# ── RUN ───────────────────────────────────────────────────────────────────────
# First: delete existing video_found / no_video_found records you want to re-match.
# Or just run as-is — it will skip already-processed races.
search_stats = run_automated_search(
    max_races=None,
    verbose=True,
    threshold=75.0,
)

Local cache: 39,486 videos across 4 channels
Races to process: 896 (limit: 896)
———————————————————————————————————————————————————————

———————————————————————————————————————————————————————
Search complete (no API quota used)
  Videos found:    0
  No video:        0
  Skipped:         896


In [28]:
# ── CELL 6: Fetch transcripts ────────────────────────────────────────────────
# Unchanged from v2. No API quota — just HTTP requests to YouTube.
# Uses deliberate delays to avoid IP blocking.

def run_transcript_fetch(
    max_transcripts: int = 50,
    delay_seconds: float = 45.0,
) -> dict:
    pending = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        if data.get('status') == 'video_found':
            pending.append((path, data))

    total = min(len(pending), max_transcripts)
    print(f'Videos pending transcript: {len(pending)}')
    print(f'Fetching up to:            {total}')
    print(f'Delay between requests:    {delay_seconds}s')
    print(f'Estimated time:            {total * delay_seconds / 60:.1f} minutes')
    print(f'{chr(8212)*55}')

    success = errors = ip_blocked = 0

    for path, data in pending[:max_transcripts]:
        label    = data['label']
        video    = data['video']
        video_id = video['video_id']

        print(f'\nFetching: {label}')
        print(f'  {video["title"][:60]}')

        transcript = fetch_transcript(video_id)

        if transcript['success']:
            data['transcript'] = {
                'snippet_count': transcript['snippet_count'],
                'raw_chars':     transcript['raw_chars'],
                'clean_chars':   transcript['clean_chars'],
                'duration_mins': transcript['duration_mins'],
                'clean_text':    transcript['clean_text'],
                'preview_start': transcript['preview_start'],
                'preview_end':   transcript['preview_end'],
            }
            data['status'] = 'transcript_saved'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars | {transcript["duration_mins"]:.1f} min')
            success += 1
        else:
            error_msg  = transcript['error']
            is_ip_block = any(kw in error_msg.lower()
                              for kw in ['blocked', 'ip', '429', 'too many'])
            if is_ip_block:
                data['status'] = 'transcript_error:ip_blocked'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                ip_blocked += 1
                print(f'\n⚠ IP blocking detected — stopping.')
                print(f'  Saved {success} transcripts before block.')
                break
            else:
                data['status'] = f'transcript_error:{error_msg[:80]}'
                with open(path, 'w', encoding='utf-8') as f:
                    json.dump(data, f, indent=2)
                errors += 1
                print(f'  ✗ {error_msg[:80]}')

        actual_delay = delay_seconds + random.uniform(-5, 10)
        actual_delay = max(20, actual_delay)
        print(f'  Waiting {actual_delay:.0f}s...')
        time.sleep(actual_delay)

    ip_blocked_total = sum(
        1 for p in RAW_DIR.glob('*.json')
        if json.load(open(p, encoding='utf-8')).get('status') == 'transcript_error:ip_blocked'
    )

    print(f'\n{chr(8212)*55}')
    print(f'Fetch complete')
    print(f'  Transcripts saved:   {success}')
    print(f'  IP blocked:          {ip_blocked}')
    print(f'  Other errors:        {errors}')
    print(f'  Still pending retry: {ip_blocked_total}')
    return {'success': success, 'ip_blocked': ip_blocked, 'errors': errors}

In [29]:
fetch_stats = run_transcript_fetch(
    max_transcripts=50,
    delay_seconds=45.0,
)

Videos pending transcript: 456
Fetching up to:            50
Delay between requests:    45.0s
Estimated time:            37.5 minutes
———————————————————————————————————————————————————————

Fetching: 2017 Giro d'Italia Stage 17
  Giro d'Italia 2017 - Stage 18 - Highlights

⚠ IP blocking detected — stopping.
  Saved 0 transcripts before block.

———————————————————————————————————————————————————————
Fetch complete
  Transcripts saved:   0
  IP blocked:          1
  Other errors:        0
  Still pending retry: 19


In [10]:
# ── CELL 7: Manual override ──────────────────────────────────────────────────
# Use to: add a video you found manually, retry IP-blocked races, or fix bad matches.
# Unchanged from v2.

def show_manual_todo() -> list:
    todo = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        s = data.get('status', '')
        if s in ('no_video_found',) or 'ip_blocked' in s:
            todo.append({'label': data['label'], 'status': s, 'path': path})
    print(f'Manual attention needed: {len(todo)}')
    for t in todo[:30]:
        print(f'  [{t["status"]}] {t["label"]}')
    if len(todo) > 30:
        print(f'  ... and {len(todo) - 30} more')
    return todo


def manual_add(video_id: str, label: str = None) -> dict:
    if label is None:
        print('Provide a label from show_manual_todo() output.')
        show_manual_todo()
        return {}

    safe_name = make_safe_name(label)
    out_path  = RAW_DIR / f'{safe_name}.json'

    # Load existing metadata if available
    race_name, race_date, stage = label, '2017-01-01', None
    if out_path.exists():
        with open(out_path, encoding='utf-8') as f:
            existing = json.load(f)
        race_name = existing.get('race_name', label)
        race_date = existing.get('race_date', '2017-01-01')
        stage     = existing.get('stage')
    else:
        # Try to find in race_index
        for _, row in race_index.iterrows():
            rn, st = parse_race_name_and_stage(row['Race Name'])
            rd     = str(row['Date'])[:10]
            if make_label(rn, rd, st) == label:
                race_name, race_date, stage = rn, rd, st
                break

    print(f'Fetching transcript for: {label}')
    print(f'  Video ID: {video_id}')
    transcript = fetch_transcript(video_id)

    if transcript['success']:
        record = {
            'label':     label,
            'race_name': race_name,
            'race_date': race_date,
            'stage':     stage,
            'video': {
                'video_id':    video_id,
                'title':       f'Manual add: {video_id}',
                'url':         f'https://youtube.com/watch?v={video_id}',
                'channel':     'manual',
                'published':   '',
                'match_score': 999,
            },
            'transcript': {
                'snippet_count': transcript['snippet_count'],
                'raw_chars':     transcript['raw_chars'],
                'clean_chars':   transcript['clean_chars'],
                'duration_mins': transcript['duration_mins'],
                'clean_text':    transcript['clean_text'],
                'preview_start': transcript['preview_start'],
                'preview_end':   transcript['preview_end'],
            },
            'status': 'transcript_saved',
        }
        with open(out_path, 'w', encoding='utf-8') as f:
            json.dump(record, f, indent=2, ensure_ascii=False)
        print(f'  ✓ Saved: {transcript["clean_chars"]:,} chars')
        return record
    else:
        print(f'  ✗ Failed: {transcript["error"]}')
        return {'error': transcript['error']}


def retry_ip_blocked(delay_seconds: float = 90.0) -> dict:
    blocked = []
    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        status   = data.get('status', '')
        video    = data.get('video') or {}
        video_id = video.get('video_id')
        is_blocked = any(kw in status.lower()
                         for kw in ['ip_blocked', 'blocking requests', 'too many requests'])
        if is_blocked and video_id:
            blocked.append((path, data))

    print(f'IP blocked races: {len(blocked)}')
    success = still_blocked = errors = 0

    for path, data in blocked:
        label    = data.get('label', path.stem)
        video_id = data['video']['video_id']
        print(f'\nRetrying: {label}')
        transcript = fetch_transcript(video_id)
        if transcript['success']:
            data['transcript'] = {k: transcript[k] for k in
                ['snippet_count','raw_chars','clean_chars','duration_mins',
                 'clean_text','preview_start','preview_end']}
            data['status'] = 'transcript_saved'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
            print(f'  ✓ {transcript["clean_chars"]:,} chars')
            success += 1
        else:
            err = transcript['error']
            if 'blocking' in err.lower() or 'ip' in err.lower():
                print('  ✗ Still IP blocked — stopping')
                still_blocked += 1
                break
            data['status'] = f'transcript_error:{err[:100]}'
            with open(path, 'w', encoding='utf-8') as f:
                json.dump(data, f, indent=2)
            errors += 1
            print(f'  ✗ {err[:80]}')

        actual = max(30, delay_seconds + random.uniform(-10, 15))
        print(f'  Waiting {actual:.0f}s...')
        time.sleep(actual)

    print(f'\nRetry done — saved: {success}, still blocked: {still_blocked}, errors: {errors}')
    return {'success': success, 'still_blocked': still_blocked, 'errors': errors}


# Uncomment to use:
# show_manual_todo()
# manual_add('VIDEO_ID_HERE', '2019 Paris-Roubaix')
# retry_ip_blocked()


In [11]:
retry_ip_blocked()

IP blocked races: 9

Retrying: 2020 Paris - Nice Stage 5
  ✗ Still IP blocked — stopping

Retry done — saved: 0, still blocked: 1, errors: 0


{'success': 0, 'still_blocked': 1, 'errors': 0}

In [12]:
# ── CELL 8: Status dashboard ─────────────────────────────────────────────────

def show_status() -> None:
    raw_files = list(RAW_DIR.glob('*.json'))
    statuses  = {}
    for path in raw_files:
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        s = data.get('status', 'unknown')
        bucket = (s if s in ('transcript_saved', 'no_video_found', 'video_found')
                  else ('ip_blocked' if 'ip_blocked' in s else 'other_error'))
        statuses[bucket] = statuses.get(bucket, 0) + 1

    extracted_count = len(list(EXTRACTED_DIR.glob('*.json')))
    remaining       = len(race_index) - len(raw_files)
    cache_size      = len(all_videos) if all_videos is not None else 0

    print('══════ Commentary Pipeline Status ══════')
    print(f'Local cache:             {cache_size:,} videos')
    print(f'Total races:             {len(race_index):,}')
    print(f'Processed:               {len(raw_files):,}')
    print(f'  ✓ Transcript saved:    {statuses.get("transcript_saved", 0)}')
    print(f'  → Video found (queue): {statuses.get("video_found", 0)}')
    print(f'  ✗ No video:            {statuses.get("no_video_found", 0)}')
    print(f'  ⚠ IP blocked:          {statuses.get("ip_blocked", 0)}')
    print(f'  ? Other errors:        {statuses.get("other_error", 0)}')
    print(f'Remaining:               {remaining:,}')
    print(f'Extracted (Claude):      {extracted_count}')
    print()
    print(f'Next steps:')
    if statuses.get('video_found', 0) > 0:
        print(f'  → Run Cell 6 to fetch {statuses["video_found"]} pending transcripts')
    if remaining > 0:
        print(f'  → Run Cell 5 to match {remaining} unprocessed races')
    if statuses.get('ip_blocked', 0) > 0:
        print(f'  → Run retry_ip_blocked() in Cell 7')


show_status()


══════ Commentary Pipeline Status ══════
Local cache:             39,486 videos
Total races:             896
Processed:               897
  ✓ Transcript saved:    434
  → Video found (queue): 0
  ✗ No video:            454
  ⚠ IP blocked:          9
  ? Other errors:        0
Remaining:               -1
Extracted (Claude):      4

Next steps:
  → Run retry_ip_blocked() in Cell 7


In [18]:
# ── CELL 9: Audit transcript quality (v3) ─────────────────────────────────
import re
import unicodedata

BAD_KEYWORDS = [
    'preview', 'riders to watch', 'ones to watch',
    'team time trial preparation', 'guide to', 'how to watch',
    'what to expect', 'top 10 riders', 'race preview',
    'stage preview', 'who to watch',
]

NON_RACE_KEYWORDS = [
    'truck tour', 'bus tour', 'kitchen truck', 'mechanics truck',
    'dutch corner', 'asks the pros', 'key climbs', 'most important climbs',
    'talking points', 'gcn show ep', 'french phrases',
    'mvps', 'super star riders', 'what we ve learnt', 'things we ve learnt',
    'custom tech', 'new bikes', 'bike tech', 'prototype bike',
    'cycling phrases', 'neutral service', 'aero tech',
    'spin vs grind', 'can pros fix', 'coffee do pro',
    'suitcase', 'leader s jersey', 'leaders jersey',
    'dirty kanza', 'cobbled classics', 'what are the',
    'how did', 'how to', 'explained',
    # Women's races
    'giro rosa', 'giro donne', 'giro women',
    # Wrong race patterns
    'tour de la provence', 'tour down under', 'tour of britain',
    'vuelta a san juan', 'santos tour',
]

STAGE_RACES = {
    "tour de france", "giro d'italia", "giro d italia",
    "vuelta a espana", "la vuelta", "criterium du dauphine",
    "tour de suisse", "tour de romandie", "paris-nice",
    "tirreno-adriatico", "volta a catalunya", "tour of the basque country",
    "itzulia basque country", "tour de pologne", "binckbank tour",
    "benelux tour", "tour of guangxi",
}

RACE_TITLE_ALIASES = {
    "vuelta a espana":               ["vuelta a espana", "la vuelta", "vuelta espana"],
    "la vuelta ciclista a espana":   ["vuelta a espana", "la vuelta", "vuelta espana"],
    "giro ditalia":                  ["giro d italia", "giro ditalia"],
    "giro d'italia":                 ["giro d italia", "giro ditalia"],
    "liege-bastogne-liege":          ["liege bastogne liege", "liege bastogne"],
    "liege - bastogne - liege":      ["liege bastogne liege", "liege bastogne"],
    "la fleche wallonne":            ["fleche wallonne"],
    "ronde van vlaanderen":          ["tour of flanders", "ronde", "vlaanderen"],
    "paris-roubaix":                 ["paris roubaix", "roubaix"],
    "milan-san remo":                ["milan san remo", "milan sanremo", "la primavera"],
    "il lombardia":                  ["lombardia", "tour of lombardy"],
    "strade bianche":                ["strade bianche"],
    "amstel gold race":              ["amstel gold", "amstel"],
    "criterium du dauphine":         ["dauphine", "dauphin"],
    "tour de suisse":                ["tour de suisse", "tour of switzerland"],
    "clasica san sebastian":         ["san sebastian", "clasica"],
    "eschborn-frankfurt":            ["eschborn frankfurt", "frankfurt"],
    "omloop het nieuwsblad":         ["omloop", "nieuwsblad"],
    "dwars door vlaanderen":         ["dwars door"],
    "e3 saxo bank classic":          ["e3 saxo", "e3 harelbeke", "e3 binckbank"],
    "gent-wevelgem":                 ["gent wevelgem"],
    "classic brugge-de panne":       ["brugge de panne", "bruges de panne"],
    "bretagne classic":              ["bretagne classic", "ouest france"],
    "tour de pologne":               ["tour de pologne"],
    "volta a catalunya":             ["volta a catalunya", "volta ciclista"],
    "paris-nice":                    ["paris nice"],
    "tirreno-adriatico":             ["tirreno adriatico"],
    "itzulia basque country":        ["itzulia", "basque country", "tour of the basque"],
    "tour of the basque country":    ["itzulia", "basque country"],
    "binckbank tour":                ["binckbank"],
    "benelux tour":                  ["benelux"],
}

MIN_SCORE = 75

# Files to always keep regardless of audit flags
# score=999 means manually provided; Paris-Nice 2017 stages 3-5 are
# good matches where NBC just didn't include the year in the title
ALWAYS_KEEP = {
    '2017_amstel_gold_race.json',
    '2017_strade_bianche.json',
    '2017_tour_de_pologne_stage_1.json',
    '2017_paris_nice_stage_3.json',
    '2017_paris_nice_stage_4.json',
    '2017_paris_nice_stage_5.json',
    '2023_la_vuelta_ciclista_a_espana_stage_15.json',
}


def _normalize_for_match(text: str) -> str:
    text = text.lower()
    text = ''.join(c for c in unicodedata.normalize('NFD', text) if unicodedata.category(c) != 'Mn')
    text = re.sub(r"[''`´'\"&;]", ' ', text)
    text = re.sub(r'[^a-z0-9 ]', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()


def _race_in_title(race_name: str, title: str) -> bool:
    race_norm  = _normalize_for_match(race_name)
    title_norm = _normalize_for_match(title)

    if race_norm in title_norm:
        return True

    words = [w for w in race_norm.split() if len(w) > 3]
    if words and sum(1 for w in words if w in title_norm) >= min(2, len(words)):
        return True

    for canonical, aliases in RACE_TITLE_ALIASES.items():
        canonical_norm = _normalize_for_match(canonical)
        if canonical_norm in race_norm or race_norm in canonical_norm:
            for alias in aliases:
                if _normalize_for_match(alias) in title_norm:
                    return True

    return False


def _stage_in_title(stage: int, title: str) -> bool:
    title_lower = title.lower()
    return any(p in title_lower for p in [
        f'stage {stage}', f'stage{stage}',
        f'étape {stage}', f'etape {stage}',
        f'tappa {stage}', f'etapa {stage}',
        f'st {stage} ',   f'st{stage} ',
    ])


def _is_stage_race(race_name: str) -> bool:
    return any(sr in race_name.lower() for sr in STAGE_RACES)


def audit_transcript_quality(
    year_filter: int = None,
    verbose: bool = True,
    require_year_in_title: bool = True,
    require_stage_in_title: bool = True,
) -> list:
    flagged = []

    for path in sorted(RAW_DIR.glob('*.json')):
        with open(path, encoding='utf-8') as f:
            data = json.load(f)

        if data.get('status') != 'transcript_saved':
            continue

        # Always keep manually provided and known-good matches
        if path.name in ALWAYS_KEEP:
            continue

        label     = data.get('label', '')
        race_year = label[:4]
        race_name = data.get('race_name', '')
        stage     = data.get('stage')

        if year_filter and not label.startswith(str(year_filter)):
            continue

        video   = data.get('video') or {}
        title   = video.get('title', '')
        score   = video.get('match_score', 999)
        channel = video.get('channel', '')

        reasons = []

        # 1 — Bad keyword
        if any(kw in title.lower() for kw in BAD_KEYWORDS):
            reasons.append('preview_keyword_in_title')

        # 2 — Non-race content or wrong race
        if any(kw in title.lower() for kw in NON_RACE_KEYWORDS):
            reasons.append('non_race_content_or_wrong_race')

        # 3 — Score too low
        if score < MIN_SCORE:
            reasons.append(f'low_score:{score:.0f}')

        # 4 — Year not in title
        if require_year_in_title and race_year not in title:
            reasons.append(f'year_{race_year}_missing_from_title')

        # 5 — Race name not in title
        if not _race_in_title(race_name, title):
            reasons.append('race_name_not_in_title')

        # 6 — Stage number missing
        if stage and _is_stage_race(race_name) and require_stage_in_title:
            if not _stage_in_title(stage, title):
                reasons.append(f'stage_{stage}_missing_from_title')

        if reasons:
            flagged.append({
                'label':   label,
                'title':   title,
                'score':   score,
                'channel': channel,
                'reasons': reasons,
                'path':    path,
            })

    print(f'Flagged as likely bad matches: {len(flagged)} / checked')
    if verbose:
        for f in flagged:
            print(f'\n  [{", ".join(f["reasons"])}]')
            print(f'  Race:  {f["label"]}')
            print(f'  Video: {f["title"]}')
            print(f'  Score: {f["score"]:.0f}  Channel: {f["channel"]}')
    return flagged


def delete_bad_matches(flagged: list, dry_run: bool = True) -> None:
    for f in flagged:
        if dry_run:
            print(f'Would reset: {f["path"].name}')
        else:
            with open(f['path'], encoding='utf-8') as fh:
                data = json.load(fh)
            clean = {
                'label':      data['label'],
                'race_name':  data.get('race_name', ''),
                'race_date':  data.get('race_date', ''),
                'stage':      data.get('stage'),
                'video':      None,
                'transcript': None,
                'status':     'no_video_found',
            }
            with open(f['path'], 'w', encoding='utf-8') as out:
                json.dump(clean, out, indent=2)
            print(f'Reset: {f["path"].name}')

In [19]:
# ── Run audit ─────────────────────────────────────────────────────────────────
bad = audit_transcript_quality(verbose=True)

# Preview what would be reset
delete_bad_matches(bad, dry_run=True)

# # Uncomment to execute:
# delete_bad_matches(bad, dry_run=False)

Flagged as likely bad matches: 0 / checked


In [20]:
# Pass 1 — reset only the genuinely bad ones (exclude manual + false positives)
SKIP_FILES = {
    '2017_amstel_gold_race.json',
    '2017_strade_bianche.json',
    '2017_tour_de_pologne_stage_1.json',
    '2017_paris_nice_stage_3.json',
    '2017_paris_nice_stage_4.json',
    '2017_paris_nice_stage_5.json',
    '2023_la_vuelta_ciclista_a_espana_stage_15.json',
}

def delete_bad_matches_filtered(flagged: list, dry_run: bool = True) -> None:
    to_reset  = [f for f in flagged if f['path'].name not in SKIP_FILES]
    to_keep   = [f for f in flagged if f['path'].name in SKIP_FILES]

    print(f'Will reset:  {len(to_reset)}')
    print(f'Will keep:   {len(to_keep)} (manually skipped)')
    print()
    for f in to_keep:
        print(f'  KEEPING: {f["path"].name}  [{", ".join(f["reasons"])}]')

    print()
    for f in to_reset:
        if dry_run:
            print(f'Would reset: {f["path"].name}')
        else:
            with open(f['path'], encoding='utf-8') as fh:
                data = json.load(fh)
            clean = {
                'label':      data['label'],
                'race_name':  data.get('race_name', ''),
                'race_date':  data.get('race_date', ''),
                'stage':      data.get('stage'),
                'video':      None,
                'transcript': None,
                'status':     'no_video_found',
            }
            with open(f['path'], 'w', encoding='utf-8') as out:
                json.dump(clean, out, indent=2)
            print(f'Reset: {f["path"].name}')


# Preview
delete_bad_matches_filtered(bad, dry_run=True)

# Execute when ready
# delete_bad_matches_filtered(bad, dry_run=False)

Will reset:  99
Will keep:   7 (manually skipped)

  KEEPING: 2017_amstel_gold_race.json  [year_2017_missing_from_title]
  KEEPING: 2017_paris_nice_stage_3.json  [year_2017_missing_from_title]
  KEEPING: 2017_paris_nice_stage_4.json  [year_2017_missing_from_title]
  KEEPING: 2017_paris_nice_stage_5.json  [year_2017_missing_from_title]
  KEEPING: 2017_strade_bianche.json  [year_2017_missing_from_title]
  KEEPING: 2017_tour_de_pologne_stage_1.json  [year_2017_missing_from_title, race_name_not_in_title, stage_1_missing_from_title]
  KEEPING: 2023_la_vuelta_ciclista_a_espana_stage_15.json  [non_race_content]

Would reset: 2017_criterium_du_dauphine_stage_2.json
Would reset: 2017_criterium_du_dauphine_stage_3.json
Would reset: 2017_criterium_du_dauphine_stage_4.json
Would reset: 2017_criterium_du_dauphine_stage_5.json
Would reset: 2017_criterium_du_dauphine_stage_6.json
Would reset: 2017_criterium_du_dauphine_stage_7.json
Would reset: 2017_criterium_du_dauphine_stage_8.json
Would reset: 201

In [21]:
delete_bad_matches_filtered(bad, dry_run=False)

Will reset:  99
Will keep:   7 (manually skipped)

  KEEPING: 2017_amstel_gold_race.json  [year_2017_missing_from_title]
  KEEPING: 2017_paris_nice_stage_3.json  [year_2017_missing_from_title]
  KEEPING: 2017_paris_nice_stage_4.json  [year_2017_missing_from_title]
  KEEPING: 2017_paris_nice_stage_5.json  [year_2017_missing_from_title]
  KEEPING: 2017_strade_bianche.json  [year_2017_missing_from_title]
  KEEPING: 2017_tour_de_pologne_stage_1.json  [year_2017_missing_from_title, race_name_not_in_title, stage_1_missing_from_title]
  KEEPING: 2023_la_vuelta_ciclista_a_espana_stage_15.json  [non_race_content]

Reset: 2017_criterium_du_dauphine_stage_2.json
Reset: 2017_criterium_du_dauphine_stage_3.json
Reset: 2017_criterium_du_dauphine_stage_4.json
Reset: 2017_criterium_du_dauphine_stage_5.json
Reset: 2017_criterium_du_dauphine_stage_6.json
Reset: 2017_criterium_du_dauphine_stage_7.json
Reset: 2017_criterium_du_dauphine_stage_8.json
Reset: 2017_giro_d_italia_stage_1.json
Reset: 2017_giro_d_

In [ ]:
# ── CELL 10: Claude tactical extraction ──────────────────────────────────────
# Reads saved transcripts and extracts structured tactical events.
# Unchanged from v2.

EXTRACTION_PROMPT = """You are analyzing cycling race commentary for {race_name}.

Extract tactically significant information from this transcript.
Focus only on information useful for pre-race analysis of future similar stages.
Ignore music, crowd noise, and generic excitement phrases.

Return a JSON object:
{{
  "race_summary": "2-3 sentence tactical summary of what happened",
  "decisive_moment": "the single most important tactical event",
  "commentary_quality": "rich|moderate|thin",
  "events": [
    {{
      "event_type": "attack|breakaway|chase|team_tactic|rider_observation|weather|terrain",
      "riders": ["LASTNAME Firstname"],
      "teams": ["Team Name"],
      "description": "what happened concisely",
      "tactical_significance": "why this matters for future analysis"
    }}
  ],
  "rider_form_signals": [
    {{
      "rider": "LASTNAME Firstname",
      "signal": "positive|negative|neutral",
      "observation": "what commentary reveals about their condition"
    }}
  ],
  "terrain_observations": [
    "observation about how terrain affected racing dynamics"
  ],
  "usable_for_rag": true
}}

Return only valid JSON, no other text.
Transcript:
{transcript}"""


def extract_one(label: str, transcript_text: str) -> dict:
    max_chars = 8000
    if len(transcript_text) > max_chars:
        half = max_chars // 2
        text = transcript_text[:half] + '\n[...middle omitted...]\n' + transcript_text[-half:]
    else:
        text = transcript_text

    response = claude.messages.create(
        model='claude-sonnet-4-5',
        max_tokens=3000,
        messages=[{'role': 'user', 'content': EXTRACTION_PROMPT.format(
            race_name=label, transcript=text
        )}]
    )
    raw = re.sub(r'```json|```', '', response.content[0].text).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {'error': 'json_parse_failed', 'raw': raw[:200]}


def run_extraction(max_extractions: int = 50, verbose: bool = True) -> dict:
    pending = [
        path for path in sorted(RAW_DIR.glob('*.json'))
        if not (EXTRACTED_DIR / path.name).exists()
        and json.load(open(path, encoding='utf-8')).get('status') == 'transcript_saved'
    ]

    total_cost = success = skipped = errors = 0
    print(f'Transcripts pending extraction: {len(pending)}')
    print(f'Processing up to: {max_extractions}')
    print(f'Est. cost: ${min(len(pending), max_extractions) * 0.02:.2f}')
    print(f'{chr(8212)*55}')

    for path in pending[:max_extractions]:
        extracted_path = EXTRACTED_DIR / path.name
        with open(path, encoding='utf-8') as f:
            data = json.load(f)
        transcript_text = data.get('transcript', {}).get('clean_text', '')
        if not transcript_text.strip():
            skipped += 1
            continue

        label = data.get('label', path.stem)
        chars = len(transcript_text)
        cost  = min(chars, 8000) / 4 / 1_000_000 * 3

        if verbose:
            print(f'Extracting: {label}')
            print(f'  Chars: {chars:,} | Est: ${cost:.4f}')

        try:
            events = extract_one(label, transcript_text)
            output = {
                'label':       label,
                'race_name':   data.get('race_name'),
                'race_date':   data.get('race_date'),
                'stage':       data.get('stage'),
                'video_title': data.get('video', {}).get('title'),
                'video_url':   data.get('video', {}).get('url'),
                'channel':     data.get('video', {}).get('channel'),
                'extraction':  events,
                'extracted_at': str(datetime.datetime.utcnow()),
            }
            with open(extracted_path, 'w', encoding='utf-8') as f:
                json.dump(output, f, indent=2, ensure_ascii=False)

            total_cost += cost
            success    += 1
            if verbose:
                quality  = events.get('commentary_quality', 'unknown')
                n_events = len(events.get('events', []))
                print(f'  ✓ Quality: {quality} | Events: {n_events}')
        except Exception as e:
            print(f'  ✗ Error: {e}')
            errors += 1

        time.sleep(0.5)

    print(f'\n{chr(8212)*55}')
    print(f'Extraction complete')
    print(f'  Extracted:   {success}')
    print(f'  Skipped:     {skipped}')
    print(f'  Errors:      {errors}')
    print(f'  Total cost:  ${total_cost:.4f}')
    return {'success': success, 'skipped': skipped, 'errors': errors, 'cost': total_cost}


# ── RUN EXTRACTION ─────────────────────────────────────────────────────────────
# extraction_stats = run_extraction(max_extractions=50, verbose=True)
print('Extraction functions ready — uncomment the line above to run')


In [ ]:
# ── CELL 11: Commentary retrieval ────────────────────────────────────────────
# Used by the agent notebook. Always returns a string — never raises.
# Unchanged from v2.

def get_commentary_context(
    race_name: str,
    race_date: str,
    stage: int = None,
    max_chars: int = 3000,
) -> str:
    label     = make_label(race_name, race_date, stage)
    safe_name = make_safe_name(label)

    extracted_path = EXTRACTED_DIR / f'{safe_name}.json'
    raw_path       = RAW_DIR       / f'{safe_name}.json'

    if extracted_path.exists():
        try:
            with open(extracted_path, encoding='utf-8') as f:
                data = json.load(f)
            ext = data.get('extraction', {})
            if 'error' not in ext:
                parts = [f'[COMMENTARY: {data.get("channel")} | {data.get("video_title","")[:55]}]']
                if ext.get('race_summary'):
                    parts.append(f'Summary: {ext["race_summary"]}')
                if ext.get('decisive_moment'):
                    parts.append(f'Decisive moment: {ext["decisive_moment"]}')
                for e in ext.get('events', [])[:5]:
                    riders = ', '.join(e.get('riders', []))
                    parts.append(f'- [{e["event_type"]}] {e["description"]}' +
                                 (f' ({riders})' if riders else ''))
                for s in ext.get('rider_form_signals', [])[:3]:
                    parts.append(f'- Form: {s["rider"]} ({s["signal"]}): {s["observation"]}')
                return '\n'.join(parts)
        except Exception:
            pass

    if raw_path.exists():
        try:
            with open(raw_path, encoding='utf-8') as f:
                data = json.load(f)
            if data.get('status') == 'transcript_saved':
                text = data.get('transcript', {}).get('clean_text', '')
                if text:
                    if len(text) > max_chars:
                        half = max_chars // 2
                        text = text[:half] + '\n[...]\n' + text[-half:]
                    channel = data.get('video', {}).get('channel', 'unknown')
                    title   = data.get('video', {}).get('title', '')[:55]
                    return f'[RAW COMMENTARY: {channel} | {title}]\n\n{text}'
        except Exception:
            pass

    return f'[NO COMMENTARY for {label}] Analysis based on structured data only.'


# Quick test
print('=== Commentary retrieval ready ===')
